# MMP-9 docking pipeline (Colab)
Runs the full pipeline from the repo: tools → receptor → ligands → docking → redocking validation → zinc check → final ranking → ChimeraX pack.

**Before you start:** the upstream QSAR output `MMP9_composite_ranking_full.csv` must be in your Drive project folder, and `MMP9/active/4H3X.pdb` must exist (or be fetched by script 05).
Replace `YOUR_USERNAME` below with your GitHub username.

In [ ]:
# 1. Mount Drive and get the code
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['MMP9_PROJECT_DIR'] = '/content/drive/MyDrive/qsar_pdb_project'
!rm -rf /content/mmp9-docking-pipeline
!git clone https://github.com/YOUR_USERNAME/mmp9-docking-pipeline.git /content/mmp9-docking-pipeline
%cd /content/mmp9-docking-pipeline/scripts

## Stage 1 - Install RDKit, Meeko, gemmi and the Vina binary

In [ ]:
!python 01_setup_and_tools.py

## Stage 2 - Receptor prep (finds the catalytic zinc, writes receptor.pdbqt and the grid box)

In [ ]:
!python 02_receptor_prep.py

### Stage 2b - Where does the 1GKC co-crystal ligand (NFH) actually sit?

In [ ]:
!python 02b_check_reference_ligand.py

## Stage 3 - Ligands (SMILES from ChEMBL → 3D → PDBQT)

In [ ]:
!python 03_ligand_prep.py

## Stage 4 - Dock all 20 with AutoDock Vina

In [ ]:
!python 04_docking.py

## Stage 5 - Redocking validation (4H3X / 10B, target RMSD < 2 Å)

In [ ]:
!python 05_redocking_validation.py

## Stage 6 - Zinc-coordination check (heavy atoms only, cutoff 2.6 Å)

In [ ]:
!python 06_zinc_interaction_check.py

## Stage 7 - Final lead selection (0.4 QSAR + 0.4 affinity + 0.2 zinc)

In [ ]:
!python 07_final_lead_selection.py

## Stage 8 - ChimeraX visualization pack
Builds single-pose PDBQTs and one `.cxc` script per compound, then zips them.

In [ ]:
!python 08_make_chimerax_pack.py
from google.colab import files
files.download(os.environ['MMP9_PROJECT_DIR'] + '/chimerax_pack.zip')

## Optional - quick 3D view inside Colab (py3Dmol)

In [ ]:
!pip install -q py3Dmol
import py3Dmol, os
P = os.environ['MMP9_PROJECT_DIR'] + '/chimerax_pack/'
cid = 'CHEMBL75164'   # change to any lead
view = py3Dmol.view(width=800, height=550)
view.addModel(open(P + 'receptor.pdbqt').read(), 'pdbqt')
view.setStyle({'model': 0}, {'cartoon': {'color': 'lightgray', 'opacity': 0.6}})
view.addModel(open(P + cid + '_best.pdbqt').read(), 'pdbqt')
view.setStyle({'model': 1}, {'stick': {'colorscheme': 'orangeCarbon'}})
view.addStyle({'model': 0, 'elem': 'Zn'}, {'sphere': {'color': 'purple', 'radius': 1.0}})
view.zoomTo({'model': 1})
view.show()